In [2]:
import pandas as pd
import duckdb 
from pathlib import Path

# =========================
# 1. 路径与连接
# =========================
con = duckdb.connect(database=":memory:")
output_dir = Path("output_aggregate_further_data")
output_dir.mkdir(parents=True, exist_ok=True)

# =========================
# 2. 读取原始数据
# =========================


### 1 用户日活

In [3]:
con.execute(
    """
    CREATE OR REPLACE TABLE dws_user_active_1d AS
    SELECT
        u.user_id,
        CAST(e.event_time AS DATE) AS stat_date,
        MIN(CAST(e.event_time AS DATE)) AS first_active_date,
        MAX(CAST(e.event_time AS DATE)) AS last_active_date,
        COUNT(*) AS event_cnt,
        COUNT(DISTINCT e.session_id) AS session_cnt,
        COUNT(DISTINCT e.task_id) AS task_touch_cnt,
        MAX(CASE WHEN e.event_type = 'create_task' THEN 1 ELSE 0 END) AS create_task_flag,
        MAX(CASE WHEN e.event_type = 'complete_task' THEN 1 ELSE 0 END) AS complete_task_flag,
        MAX(CASE WHEN e.event_type = 'delete_task' THEN 1 ELSE 0 END) AS delete_task_flag
    FROM read_parquet('output_aggregate_data/00_v_todo_event_raw_clean.parquet') e
    LEFT JOIN read_parquet('output_data/01_dim_user.parquet') u
        ON e.user_id = u.user_id
    GROUP BY u.user_id, CAST(e.event_time AS DATE);
    """,
    []
)


print("dws_user_active_1d preview:")

dws_user_active_1d = con.execute("SELECT * FROM dws_user_active_1d").fetchdf()
dws_user_active_1d


con.execute(
    "COPY dws_user_active_1d TO 'output_aggregate_further_data/01_dws_user_active_1d.parquet' (FORMAT PARQUET)"
)

dws_user_active_1d preview:


### 2 用户行为汇总（每用户）

In [4]:
con.execute(
    """
    CREATE OR REPLACE TABLE dws_user_behavior_summary AS
    WITH user_event AS (
        SELECT
            user_id,
            COUNT(*) AS total_event_cnt,
            COUNT(DISTINCT CAST(event_time AS DATE)) AS active_day_cnt,
            MAX(CAST(event_time AS DATE)) AS last_active_date,
            COUNT(DISTINCT CASE WHEN event_type = 'create_task' THEN task_id END) AS create_task_cnt,
            COUNT(DISTINCT CASE WHEN event_type = 'complete_task' THEN task_id END) AS complete_task_cnt,
            COUNT(DISTINCT CASE WHEN event_type = 'delete_task' THEN task_id END) AS delete_task_cnt
        FROM read_parquet('output_aggregate_data/00_v_todo_event_raw_clean.parquet')
        GROUP BY user_id
    ),
    user_lifecycle AS (
        SELECT
            task_id,
            user_id,
            complete_duration_seconds,
            is_completed,
            is_overdue
        FROM read_parquet('output_aggregate_data/02_dwd_task_lifecycle.parquet')
    )
    SELECT
        ue.user_id,
        ue.total_event_cnt,
        ue.active_day_cnt,
        ue.last_active_date,
        COALESCE(ue.create_task_cnt, 0) AS create_task_cnt,
        COALESCE(ue.complete_task_cnt, 0) AS complete_task_cnt,
        COALESCE(ue.delete_task_cnt, 0) AS delete_task_cnt,
        CASE
            WHEN COALESCE(ue.create_task_cnt, 0) > 0
            THEN CAST(ue.complete_task_cnt AS DOUBLE) / ue.create_task_cnt
            ELSE 0
        END AS completion_rate,
        AVG(ul.complete_duration_seconds) AS avg_complete_duration_seconds
    FROM user_event ue
    LEFT JOIN user_lifecycle ul
        ON ue.user_id = ul.user_id
    GROUP BY
        ue.user_id,
        ue.total_event_cnt,
        ue.active_day_cnt,
        ue.last_active_date,
        ue.create_task_cnt,
        ue.complete_task_cnt,
        ue.delete_task_cnt;
    """,
    []
)

print("dws_user_behavior_summary preview:")

dws_user_behavior_summary = con.execute("SELECT * FROM dws_user_behavior_summary").fetchdf()
dws_user_behavior_summary


con.execute(
    "COPY dws_user_behavior_summary TO 'output_aggregate_further_data/02_dws_user_behavior_summary.parquet' (FORMAT PARQUET)"
)



dws_user_behavior_summary preview:


### 3 任务每日汇总

In [5]:
con.execute(
    """
        CREATE OR REPLACE TABLE dws_task_summary_1d AS
        SELECT
            CAST(create_time AS DATE) AS stat_date,
            COUNT(*) AS task_created_cnt,
            SUM(CASE WHEN is_completed = TRUE THEN 1 ELSE 0 END) AS task_completed_cnt,
            SUM(CASE WHEN is_overdue = TRUE THEN 1 ELSE 0 END) AS overdue_cnt,
            SUM(CASE WHEN is_completed = TRUE THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS completion_rate,
            AVG(CASE WHEN complete_duration_seconds IS NOT NULL THEN complete_duration_seconds END) AS avg_complete_duration_seconds
        FROM read_parquet('output_aggregate_data/02_dwd_task_lifecycle.parquet')
        WHERE create_time IS NOT NULL
        GROUP BY CAST(create_time AS DATE)
        ORDER BY stat_date;
    """,
    []
)

print("dws_task_summary_1d preview:")

dws_task_summary_1d = con.execute("SELECT * FROM dws_task_summary_1d").fetchdf()
dws_task_summary_1d


con.execute(
    "COPY dws_task_summary_1d TO 'output_aggregate_further_data/03_dws_task_summary_1d.parquet' (FORMAT PARQUET)"
)

dws_task_summary_1d preview:


### 4 事件类型每日汇总

In [6]:
con.execute(
    """
        CREATE OR REPLACE TABLE dws_event_type_summary_1d AS
        SELECT
            CAST(event_time AS DATE) AS stat_date,
            event_type,
            COUNT(*) AS event_cnt,
            ROUND(COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY CAST(event_time AS DATE)), 4) AS event_ratio,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY CAST(event_time AS DATE)), 2) AS event_ratio_pct
        FROM read_parquet('output_aggregate_data/00_v_todo_event_raw_clean.parquet')
        GROUP BY CAST(event_time AS DATE), event_type
        ORDER BY stat_date, event_cnt DESC;
    """,
    []
)
print("dws_event_type_summary_1d preview:")

dws_event_type_summary_1d = con.execute("SELECT * FROM dws_event_type_summary_1d").fetchdf()
dws_event_type_summary_1d


con.execute(
    "COPY dws_event_type_summary_1d TO 'output_aggregate_further_data/04_dws_event_type_summary_1d.parquet' (FORMAT PARQUET)"
)

dws_event_type_summary_1d preview:


### 5 按小时活跃分布

In [7]:
con.execute(
    """
        CREATE OR REPLACE TABLE dws_hour_active_summary AS
        SELECT
            CAST(event_time AS DATE) AS stat_date,
            EXTRACT(HOUR FROM event_time) AS active_hour,
            COUNT(*) AS event_cnt,
            COUNT(DISTINCT user_id) AS active_user_cnt
        FROM read_parquet('output_aggregate_data/00_v_todo_event_raw_clean.parquet')
        GROUP BY CAST(event_time AS DATE), EXTRACT(HOUR FROM event_time)
        ORDER BY stat_date, active_hour;
    """,
    []
)

print("dws_hour_active_summary preview:")

dws_hour_active_summary = con.execute("SELECT * FROM dws_hour_active_summary").fetchdf()
dws_hour_active_summary

con.execute(
    "COPY dws_hour_active_summary TO 'output_aggregate_further_data/05_dws_hour_active_summary.parquet' (FORMAT PARQUET)"
)

dws_hour_active_summary preview:


### 6 渠道 + 区域汇总

In [8]:
con.execute(
    """
        CREATE OR REPLACE TABLE dws_channel_region_summary AS
        SELECT
            CAST(e.event_time AS DATE) AS stat_date,
            u.register_channel,
            u.province,
            u.city,
            COUNT(DISTINCT e.user_id) AS active_user_cnt,
            COUNT(*) AS event_cnt
        FROM read_parquet('output_aggregate_data/00_v_todo_event_raw_clean.parquet') e
        LEFT JOIN read_parquet('output_data/01_dim_user.parquet') u
            ON e.user_id = u.user_id
        GROUP BY
            CAST(e.event_time AS DATE),
            u.register_channel,
            u.province,
            u.city
        ORDER BY stat_date DESC, active_user_cnt DESC;
    """,
    []
)

print("dws_channel_region_summary preview:")

dws_channel_region_summary = con.execute("SELECT * FROM dws_channel_region_summary").fetchdf()
dws_channel_region_summary

con.execute(
    "COPY dws_channel_region_summary TO 'output_aggregate_further_data/06_dws_channel_region_summary.parquet' (FORMAT PARQUET)"
)


dws_channel_region_summary preview:


### 其他内容
### 7 新老用户

In [9]:
con.execute(
    """
    CREATE OR REPLACE TABLE dws_new_old_user_summary AS
    WITH first_active AS (
        SELECT
            user_id,
            MIN(CAST(event_time AS DATE)) AS first_active_date
        FROM read_parquet('output_aggregate_data/00_v_todo_event_raw_clean.parquet')
        GROUP BY user_id
    )
    SELECT
        d.active_date AS stat_date,
        SUM(CASE WHEN f.first_active_date = d.active_date THEN 1 ELSE 0 END) AS new_user_cnt,
        SUM(CASE WHEN f.first_active_date <> d.active_date THEN 1 ELSE 0 END) AS old_user_cnt
    FROM read_parquet('output_aggregate_data/03_dwd_user_active_detail.parquet') d
    LEFT JOIN first_active f
        ON d.user_id = f.user_id
    GROUP BY d.active_date
    ORDER BY d.active_date;
    """,
    []
)

con.execute(
    "COPY dws_new_old_user_summary TO 'output_aggregate_further_data/07_dws_new_old_user_summary.parquet' (FORMAT PARQUET)"
)


### 8 留存

In [10]:
con.execute("""
CREATE OR REPLACE TABLE dws_user_retain_summary AS
WITH user_first_day AS (
    SELECT
        user_id,
        MIN(active_date) AS first_active_date
    FROM read_parquet('output_aggregate_data/03_dwd_user_active_detail.parquet')
    GROUP BY user_id
),
retention_base AS (
    SELECT
        ufd.user_id,
        ufd.first_active_date,
        d.active_date AS active_date
    FROM user_first_day ufd
    LEFT JOIN read_parquet('output_aggregate_data/03_dwd_user_active_detail.parquet') d
        ON ufd.user_id = d.user_id
)
SELECT
    first_active_date,
    COUNT(DISTINCT user_id) AS total_new_users,
    COUNT(DISTINCT CASE WHEN DATE_DIFF('day', first_active_date, active_date) = 1 THEN user_id END) AS day1_retained_users,
    COUNT(DISTINCT CASE WHEN DATE_DIFF('day', first_active_date, active_date) = 7 THEN user_id END) AS day7_retained_users,
    COUNT(DISTINCT CASE WHEN DATE_DIFF('day', first_active_date, active_date) = 30 THEN user_id END) AS day30_retained_users
FROM retention_base
GROUP BY first_active_date
ORDER BY first_active_date;
""",
[])


con.execute(
    "COPY dws_user_retain_summary TO 'output_aggregate_further_data/08_dws_user_retain_summary.parquet' (FORMAT PARQUET)"
)

### 9 功能渗透率

In [ ]:
con.execute(
    """
    CREATE OR REPLACE TABLE dws_user_feature_usage_summary AS
    SELECT
        CAST(e.event_time AS DATE) AS stat_date,
        AVG(CASE WHEN tl.has_reminder = TRUE THEN 1.0 ELSE 0.0 END) AS remind_feature_usage_rate,
        AVG(CASE WHEN tl.task_id IS NOT NULL AND tl.has_subtask = TRUE THEN 1.0 ELSE 0.0 END) AS subtask_usage_rate,
        AVG(CASE WHEN tl.due_date IS NOT NULL THEN 1.0 ELSE 0.0 END) AS due_date_setting_rate
    FROM read_parquet('output_aggregate_data/00_v_todo_event_raw_clean.parquet') e
    LEFT JOIN read_parquet('output_aggregate_data/02_dwd_task_lifecycle.parquet') tl
        ON e.task_id = tl.task_id
    GROUP BY CAST(e.event_time AS DATE);
    """,
    []
)


con.execute(
    "COPY dws_user_feature_usage_summary TO 'output_aggregate_further_data/09_dws_user_feature_usage_summary.parquet' (FORMAT PARQUET)"
)
